In [1]:
import os
import csv

In [5]:
dataset_dir = "../dataset" 
output_csv = "dataset.csv" 
classes = { "person": 1, "other": 0 }
rows = []

In [6]:
for class_name, label in classes.items():
    folder = os.path.join(dataset_dir, class_name)
    for filename in os.listdir(folder):
        if filename.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
            filepath = os.path.join(folder, filename)
            rows.append([filepath, label])


In [7]:
with open(output_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["path", "label"])
    writer.writerows(rows)

print("Готово!")


Готово!


In [9]:
import numpy as np
import cv2, os
from tensorflow import keras

In [10]:
# Параметры
IMG_HEIGHT = 96
IMG_WIDTH = 96
DATA_DIR = "../dataset"  # путь к папке с поддиректориями классов

# Загрузка и подготовка данных
classes = ["person", "other"]
X, y = [], []
for label, cls in enumerate(classes):
    folder = os.path.join(DATA_DIR, cls)
    for fname in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, fname), cv2.IMREAD_GRAYSCALE)
        if img is None: 
            continue
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))  # масштабируем к 96x96
        X.append(img.astype(np.float32) / 255.0)        # нормализуем пиксели 0..1
        y.append(label)
X = np.array(X).reshape(-1, IMG_HEIGHT, IMG_WIDTH, 1)
y = np.array(y)
print("Loaded", X.shape, "data samples.")


FileNotFoundError: [WinError 3] Системе не удается найти указанный путь: 'data\\person'

In [ ]:

# Разделение на обучение и тест (например 80/20)
indices = np.arange(len(X))
np.random.shuffle(indices)
train_split = int(0.8 * len(X))
X_train, X_test = X[indices[:train_split]], X[indices[train_split:]]
y_train, y_test = y[indices[:train_split]], y[indices[train_split:]]

# Простая CNN модель
model = keras.models.Sequential([
    keras.layers.Conv2D(8, (3,3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 1)),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Conv2D(16, (3,3), activation='relu'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Flatten(),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(len(classes), activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Обучение (например, 10 эпох)
model.fit(X_train, y_train, epochs=10, batch_size=8, validation_data=(X_test, y_test))
# Оценка точности на тесте
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test accuracy:", test_acc)